In [1]:
# Clasificación multiclase con embeddings semánticos
# 1. Importar librerías
import json
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import numpy as np

In [2]:
# 2. Definir constantes y funciones principales
CLASES = ['peliculas', 'restaurantes', 'productos', 'servicios', 'hoteles']
RANDOM_STATE = 42

def cargar_corpus(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    reviews = [item['texto'] for item in data['reviews']]
    categorias = [item['categoria'] for item in data['reviews']]
    return reviews, categorias

def split_datos(reviews, categorias):
    return train_test_split(
        reviews, categorias, test_size=0.3, random_state=RANDOM_STATE, stratify=categorias
    )

def entrenar_modelo(reviews_train, y_train):
    encoder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    X_train = encoder.encode(reviews_train, show_progress_bar=True)
    scaler = StandardScaler()
    X_train_norm = scaler.fit_transform(X_train)
    modelo = LogisticRegression(random_state=RANDOM_STATE, solver='lbfgs', max_iter=1000)
    modelo.fit(X_train_norm, y_train)
    return encoder, scaler, modelo

def evaluar_modelo(modelo, scaler, encoder, reviews_test, y_test):
    X_test = encoder.encode(reviews_test, show_progress_bar=True)
    X_test_norm = scaler.transform(X_test)
    y_pred = modelo.predict(X_test_norm)
    print(classification_report(y_test, y_pred))
    return y_pred

def confusion_matrix_manual(y_true, y_pred, pos_label='positivo'):
    TP = sum(t == pos_label and p == pos_label for t, p in zip(y_true, y_pred))
    TN = sum(t != pos_label and p != pos_label for t, p in zip(y_true, y_pred))
    FP = sum(t != pos_label and p == pos_label for t, p in zip(y_true, y_pred))
    FN = sum(t == pos_label and p != pos_label for t, p in zip(y_true, y_pred))
    return TP, TN, FP, FN

def accuracy(y_true, y_pred):
    TP, TN, FP, FN = confusion_matrix_manual(y_true, y_pred)
    return (TP + TN) / (TP + TN + FP + FN)

def precision(y_true, y_pred, pos_label='positivo'):
    TP, _, FP, _ = confusion_matrix_manual(y_true, y_pred, pos_label)
    return TP / (TP + FP) if (TP + FP) > 0 else 0.0

def recall(y_true, y_pred, pos_label='positivo'):
    TP, _, _, FN = confusion_matrix_manual(y_true, y_pred, pos_label)
    return TP / (TP + FN) if (TP + FN) > 0 else 0.0

def f1(y_true, y_pred, pos_label='positivo'):
    p = precision(y_true, y_pred, pos_label)
    r = recall(y_true, y_pred, pos_label)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

def confusion_matrix_multiclase(y_true, y_pred, clases):
    ancho = 14
    cabecera = ' ' * ancho + ''.join(c[:ancho].ljust(ancho) for c in clases)
    print(cabecera)
    print('-' * (ancho * (len(clases) + 1)))
    for real in clases:
        fila = real[:ancho].ljust(ancho)
        for pred in clases:
            count = sum(t == real and p == pred for t, p in zip(y_true, y_pred))
            fila += str(count).ljust(ancho)
        print(fila)

def metricas_por_clase(y_test, y_pred):
    print(f"{'Categoría':<15} {'Precision':>10} {'Recall':>10} {'F1':>10}")
    print('-' * 48)
    f1_por_clase = []
    for c in CLASES:
        p = precision(y_test, y_pred, pos_label=c)
        r = recall(y_test, y_pred, pos_label=c)
        f = f1(y_test, y_pred, pos_label=c)
        f1_por_clase.append(f)
        print(f'{c:<15} {p:>10.2f} {r:>10.2f} {f:>10.2f}')
    print('-' * 48)
    macro_f1 = sum(f1_por_clase) / len(f1_por_clase)
    print(f"{'Macro F1':<15} {'':>10} {'':>10} {macro_f1:>10.2f}")
    print()
    print(f'Accuracy global: {accuracy(y_test, y_pred):.2f}')

def mostrar_matriz_confusion(y_test, y_pred):
    confusion_matrix_multiclase(y_test, y_pred, CLASES)

def predecir_nuevos(modelo, encoder):
    nuevas_reviews = [
        'La actuación fue brillante y la historia muy emotiva',
        'El sushi estaba fresco y el servicio impecable',
        'El producto llegó en perfectas condiciones y funciona genial',
        'Tardaron semanas en responder y no solucionaron el problema',
        'Habitación limpia, cama cómoda y muy buena ubicación',
    ]
    preds = modelo.predict(encoder.encode(nuevas_reviews))
    for texto, pred in zip(nuevas_reviews, preds):
        print(f'  [{pred.upper():<13}]  {texto}')




In [3]:
# IMPORTANTE: subir el archivo corpus_sentimiento_reviews.json en Colab antes de ejecutar
from google.colab import files
uploaded = files.upload()

reviews, categorias = cargar_corpus("corpus_sentimiento_reviews.json")

print(f'Total de reseñas: {len(reviews)}')
for c in CLASES:
    print(f'  {c:<15}: {categorias.count(c):>3}')

reviews_train, reviews_test, y_train, y_test = split_datos(reviews, categorias)

encoder, scaler, modelo = entrenar_modelo(reviews_train, y_train)

y_pred = evaluar_modelo(modelo, scaler, encoder, reviews_test, y_test)

metricas_por_clase(y_test, y_pred)
mostrar_matriz_confusion(y_test, y_pred)

predecir_nuevos(modelo, encoder)

Saving corpus_sentimiento_reviews.json to corpus_sentimiento_reviews.json
Total de reseñas: 150
  peliculas      :  36
  restaurantes   :  28
  productos      :  32
  servicios      :  28
  hoteles        :  26


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

              precision    recall  f1-score   support

     hoteles       0.38      0.38      0.38         8
   peliculas       1.00      0.64      0.78        11
   productos       0.62      0.50      0.56        10
restaurantes       0.67      1.00      0.80         8
   servicios       0.60      0.75      0.67         8

    accuracy                           0.64        45
   macro avg       0.65      0.65      0.64        45
weighted avg       0.68      0.64      0.64        45

Categoría        Precision     Recall         F1
------------------------------------------------
peliculas             1.00       0.64       0.78
restaurantes          0.67       1.00       0.80
productos             0.62       0.50       0.56
servicios             0.60       0.75       0.67
hoteles               0.38       0.38       0.38
------------------------------------------------
Macro F1                                    0.64

Accuracy global: 1.00
              peliculas     restaurantes  produ